In [82]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


In [83]:
df = pd.read_csv("01 covid_toy.csv")
display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [84]:
print(df.dtypes)

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [85]:
numeric_columns = df.select_dtypes(include=np.number).columns
print("Numerical Columns:")
print(list(numeric_columns))

Numerical Columns:
['age', 'fever']


In [86]:
categorical_columns = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:")
print(list(categorical_columns))

Categorical Columns:
['gender', 'cough', 'city', 'has_covid']


In [87]:
categorical_columns = ["gender", "cough", "city", "has_covid"]
for col in categorical_columns:
    df[col] = df[col].astype("object")

print("Categorical columns:", list(df.select_dtypes(include=["object"]).columns))
print("Numerical columns:", list(df.select_dtypes(include=np.number).columns))

Categorical columns: ['gender', 'cough', 'city', 'has_covid']
Numerical columns: ['age', 'fever']


In [88]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [89]:
numeric_columns = df.select_dtypes(include=np.number).columns
scaler = MinMaxScaler()
scaled_data = pd.DataFrame(
    scaler.fit_transform(df[numeric_columns]),
    columns=numeric_columns,
    index=df.index
)
display(scaled_data.head())

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [90]:
categorical_columns = ["gender", "cough", "city"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_array = encoder.fit_transform(df[categorical_columns])
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_data = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)
display(encoded_data.head())

,gender_Female,gender_Male,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [91]:
X = df.drop(columns=["has_covid"])
y = df["has_covid"]

print("Feature columns:", list(X.columns))
print("Target column:", y.name)
print("Target values:")
print(y.value_counts())

Feature columns: ['age', 'gender', 'fever', 'cough', 'city']
Target column: has_covid
Target values:
has_covid
No     55
Yes    45
Name: count, dtype: int64


In [92]:
numeric_features = ["age", "fever"]
categorical_features = ["gender", "cough", "city"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ],
    verbose_feature_names_out=False
)

In [93]:
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()
X_processed = pd.DataFrame(X_processed, columns=feature_names, index=X.index)

selector = SelectKBest(score_func=f_classif, k=min(3, X_processed.shape[1]))
X_selected_array = selector.fit_transform(X_processed, y)
selected_features = X_processed.columns[selector.get_support()]

X_selected = pd.DataFrame(X_selected_array, columns=selected_features, index=X.index)
print("Selected features:")
print(list(selected_features))
display(X_selected.head())

Selected features:
['fever', 'cough_Mild', 'cough_Strong']


,fever,cough_Mild,cough_Strong
0,0.833333,1.0,0.0
1,0.333333,1.0,0.0
2,0.500000,1.0,0.0
3,0.000000,1.0,0.0
4,0.500000,1.0,0.0
